# LC10 — Time series models: a SARIMA baseline for Svedala (self-paced, ~45 min)

Before anyone is allowed to throw machine learning at a forecasting problem (Module 3), they earn the right by building the classical baseline it must beat. This notebook fits a seasonal ARIMA to the Svedala ZON_MITT load and — more importantly — evaluates it the only honest way: **walk-forward, against persistence**. Lab 7 repeats this working method on your own Lab 6 series; material here appears in **Quiz 3**.

In [ ]:
# Install exactly what this notebook uses.
%pip install pandas numpy statsmodels matplotlib pyarrow --quiet
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
df = pd.read_parquet("../data/svedala-year/svedala_hourly.parquet")
y = df["ZON_MITT"].interpolate(limit=3).dropna()
y.plot(figsize=(10,2.5), title="ZON_MITT, one year");

In [ ]:
from pathlib import Path
# Guard: this notebook expects to run from the notebooks/ folder of a clone of
# the course repository — the datasets live one level up in ../data/.
# Failing here, early and clearly, beats a confusing FileNotFoundError later.
assert Path("../data").exists(), (
    "Course data folder not found. Clone KTH-EG2140/course-material and open "
    "this notebook from its notebooks/ folder.")

## 1. Look for structure: the ACF

A load series is nothing but structure — daily cycle, weekly cycle, weather trend. The autocorrelation function shows it directly:

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
plot_acf(y, lags=72, ax=ax[0]); ax[0].set_title("ACF — see the 24 h wave")
plot_acf(y.diff(24).dropna(), lags=72, ax=ax[1]); ax[1].set_title("after seasonal differencing")
plt.tight_layout()

The 24-hour wave dominates — so the model must contain it. That is what the *seasonal* part of SARIMA(p,d,q)(P,D,Q)ₛ does: model the change **from yesterday's same hour**, then let ARMA terms mop up what remains.

## 2. Fit — on six weeks, honestly split

Never fit on data you will evaluate on. Train: six winter weeks. Test: the week after.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
train = y.loc["2025-01-06":"2025-02-15"]
test  = y.loc["2025-02-16":"2025-02-22"]
model = SARIMAX(train, order=(1,0,1), seasonal_order=(1,1,1,24),
                enforce_stationarity=False, enforce_invertibility=False)
fit = model.fit(disp=False)
print(fit.summary().tables[0])

## 3. Evaluate like an operator: walk-forward, 24 h ahead

An operator forecasts tomorrow every day, learning nothing from the future. Simulate exactly that: for each test day, forecast 24 h ahead, then reveal that day and continue. And always alongside the humblest competitor there is — **persistence**: "tomorrow = today".

In [ ]:
horizon = 24
preds, persist, actual = [], [], []
state = fit
# Walk-forward loop: for each test day, forecast 24 h ahead from what is known,
# THEN reveal that day (state.append) so the next iteration may use it.
# .append updates the model state with new data without refitting parameters —
# cheap, and exactly what an operator's system does overnight.
for day in pd.date_range(test.index[0], test.index[-1], freq="24h"):
    fc = state.forecast(horizon)
    chunk = y.loc[day : day + pd.Timedelta(hours=horizon-1)]
    preds.append(pd.Series(fc.values[:len(chunk)], index=chunk.index))
    persist.append(y.shift(24).loc[chunk.index])
    actual.append(chunk)
    state = state.append(chunk)          # reveal the day, don't refit
pred = pd.concat(preds); pers = pd.concat(persist); act = pd.concat(actual)
mae_s = (pred - act).abs().mean(); mae_p = (pers - act).abs().mean()
print(f"walk-forward MAE, 24 h ahead:  SARIMA {mae_s:.0f} MW   persistence {mae_p:.0f} MW")
print(f"skill vs persistence: {100*(1-mae_s/mae_p):+.0f}%")

In [ ]:
ax = act.plot(figsize=(10,3), label="actual")
pred.plot(ax=ax, label="SARIMA"); pers.plot(ax=ax, label="persistence", alpha=0.5)
ax.legend(); ax.set_title("Test week, 24 h ahead walk-forward"); ax.set_ylabel("MW");

Read the skill number, not the plot: a model that cannot beat "tomorrow = today" has learned nothing worth deploying — and on calm weeks persistence is *hard* to beat. This number is the bar every Module 3 learner must clear, on exactly this evaluation. The things SARIMA cannot see — temperature swings, holidays — are precisely where covariates and learned models will earn their keep.

## Self-check

In [ ]:
assert mae_s < mae_p, "SARIMA should beat persistence on this winter week"
assert mae_s < 0.10 * act.mean(), "24h MAE above 10% of mean load — something is off"
print(f"ALL OK — baseline set: SARIMA {mae_s:.0f} MW vs persistence {mae_p:.0f} MW. Lab 7 makes this yours.")